# Dev tools — le kit de maintenance du dépôt (`argumentation_analysis/utils/dev_tools/`)

Asset CoursIA #1961 Phase 1. Douze modules d'outils de développement (plus le `__init__.py`
d'exports), dont **huit noyau** : encodage, syntaxe/tokens, environnement Java et dépendances,
mapping package→module, imports par chemin, apostrophes de docstrings, références de
répertoires, couverture, formatage autopep8.

**Corpus-free** : aucun texte du dataset, aucun appel LLM, aucune JVM. Les commandes externes
(`java -version`, `autopep8`) sont **stubées** pour la reproductibilité ; tout ce qui est affirmé
ici a été **mesuré** sur les vrais modules et rejoué en assert contre
`dev_tools_examples.json`.

In [1]:
import json, os, re, shutil, sys, tempfile
from pathlib import Path

EXAMPLES = json.loads(Path("docs/coursia_contrib/dev_tools_examples.json").read_text(encoding="utf-8"))
assert EXAMPLES["asset"] == "dev_tools"
print("sections:", [k for k in EXAMPLES if not k.startswith("meta")][:14])

sections: ['asset', 'issue', 'phase', 'inventory', 'encoding_cases', 'java_env_cases', 'java_env_rule', 'deps_cases', 'map_module_cases', 'import_cases', 'apostrophe_cases', 'syntax_cases', 'dir_reference_cases', 'coverage_cases']


## 1. Inventaire mesuré

In [2]:
from argumentation_analysis.utils.dev_tools import (
    code_formatting_utils, code_validation, coverage_utils, encoding_utils,
    env_checks, format_utils, import_testing_utils, project_structure_utils)

DEV_TOOLS_DIR = Path("argumentation_analysis") / "utils" / "dev_tools"
mods = sorted(p.name for p in DEV_TOOLS_DIR.glob("*.py") if p.name != "__init__.py")
lines = {m: len((DEV_TOOLS_DIR / m).read_text(encoding="utf-8").splitlines()) for m in mods}
inv = EXAMPLES["inventory"]
assert len(mods) == inv["total_modules"], (len(mods), inv["total_modules"])
assert lines == inv["module_lines"], "line counts drifted"
assert inv["core_eight"] == [m for m in sorted(inv["module_lines"]) if m in inv["core_eight"]]
print(f"{len(mods)} modules, noyau = {len(inv['core_eight'])}, "
      f"non ciblés = {[m for m in inv['untargeted']]}")
print("total lignes noyau:", sum(lines[m] for m in inv["core_eight"]))

12 modules, noyau = 8, non ciblés = ['refactoring_utils.py', 'repair_utils.py', 'reporting_utils.py', 'verification_utils.py']
total lignes noyau: 2215


## 2. Encodage — la chaîne utf-8 → windows-1252 → cp1252 → iso-8859-1 → latin-1

`fix_file_encoding` essaie chaque encodage dans l'ordre et réécrit en UTF-8. Le piège :
**latin-1 mappe chaque octet**, donc la chaîne n'échoue *jamais* — des octets invalides
partout « réussissent » en mojibake.

In [3]:
d = Path(tempfile.mkdtemp(prefix="enc_"))

p = d / "utf8_ok.txt"; p.write_text("café déjà vu\n", encoding="utf-8")
ok = encoding_utils.fix_file_encoding(str(p))
assert ok is True and p.read_text(encoding="utf-8") == "café déjà vu\n"

p = d / "cp1252_src.txt"; p.write_bytes("café".encode("cp1252"))
ok = encoding_utils.fix_file_encoding(str(p))
assert ok is True and p.read_text(encoding="utf-8") == "café"

p = d / "weird_bytes.txt"; p.write_bytes(b"\x81\x8d\x8f\x90\x9d")
ok = encoding_utils.fix_file_encoding(str(p))
assert ok is True  # latin-1 « réussit » quand même
assert p.read_text(encoding="utf-8") == "\x81\x8d\x8f\x90\x9d"

assert encoding_utils.fix_file_encoding(str(d / "nope.txt")) is False
cases = {c["name"]: c for c in EXAMPLES["encoding_cases"]}
assert cases["utf8_file_passes_through"]["returns"] is True
assert cases["latin1_always_decodes"]["returns"] is True
assert cases["missing_file_is_false"]["returns"] is False
print("utf-8 strict d'abord, cp1252 récupéré, latin-1 aveugle : mesuré")

utf-8 strict d'abord, cp1252 récupéré, latin-1 aveugle : mesuré


In [4]:
root = Path(tempfile.mkdtemp(prefix="scan_"))
(root / "good.py").write_text("print('éàç')\n", encoding="utf-8")
(root / "bad.py").write_bytes("print('éàç')\n".encode("cp1252"))
(root / "venv").mkdir()
(root / "venv" / "ignored.py").write_bytes("print('éàç')\n".encode("cp1252"))
(root / "notes.txt").write_bytes("print('éàç')\n".encode("cp1252"))
flagged = encoding_utils.check_project_python_files_encoding(str(root))
assert sorted(Path(f).name for f in flagged) == ["bad.py"]
sc = {c["name"]: c for c in EXAMPLES["encoding_cases"]}["project_scan_flags_only_utf8_broken_py"]
assert sc["flagged_basenames"] == ["bad.py"]
print("scan projet : seuls les .py non-UTF-8 hors répertoires exclus sont signalés")

scan projet : seuls les .py non-UTF-8 hors répertoires exclus sont signalés


## 3. `check_java_environment` — un ET strict, pas un OU

Le verdict final est `java_ok AND java_home_valid` : **java dans le PATH sans JAVA_HOME
valide → False**. Ici `java -version` est stubé pour isoler la logique.

In [5]:
d = Path(tempfile.mkdtemp(prefix="jdk_"))
fake_jdk = d / "jdk"; (fake_jdk / "bin").mkdir(parents=True)
exe = fake_jdk / "bin" / ("java.exe" if os.name == "nt" else "java")
exe.write_text("", encoding="utf-8")

def scenario(stub_ok, java_home):
    real_run, real_home = env_checks._run_command, os.environ.get("JAVA_HOME")
    try:
        env_checks._run_command = (lambda cmd, cwd=None: (0, "", 'openjdk version "11"')) if stub_ok \
            else (lambda cmd, cwd=None: (-1, "", "FileNotFoundError: java"))
        if java_home is None: os.environ.pop("JAVA_HOME", None)
        else: os.environ["JAVA_HOME"] = str(java_home)
        return env_checks.check_java_environment()
    finally:
        env_checks._run_command = real_run
        if real_home is None: os.environ.pop("JAVA_HOME", None)
        else: os.environ["JAVA_HOME"] = real_home

measured = {
    "path_only_no_home":        scenario(True, None),
    "home_and_path":            scenario(True, fake_jdk),
    "home_without_bin_exe":     scenario(True, d),
    "java_not_found_valid_home": scenario(False, fake_jdk),
}
for c in EXAMPLES["java_env_cases"]:
    assert measured[c["name"]] is c["verdict"], (c["name"], measured[c["name"]])
assert measured["path_only_no_home"] is False
assert measured["home_and_path"] is True
print("verdict = java_ok AND java_home_valid — mesuré sur les 4 quadrants")

verdict = java_ok AND java_home_valid — mesuré sur les 4 quadrants


## 4. `check_python_dependencies` — verdict booléen, lignes spéciales ignorées

Les lignes `-e`, `git+`, `.git@` et `-r` sont **sautées** (elles ne font pas échouer).
Un fichier vide/commentaires → **True**. Une contrainte impossible à parser est
*récupérée* par extraction du nom seul… mais marque le verdict **False** (la perte de
contrainte est un échec, pas un avertissement).

In [6]:
d = Path(tempfile.mkdtemp(prefix="deps_"))
def check(name, content):
    p = d / f"{name}.txt"; p.write_text(content, encoding="utf-8")
    return env_checks.check_python_dependencies(str(p))

measured = {
    "all_installed": check("ok", "pytest\npackaging>=20\n"),
    "missing_package": check("miss", "pytest\nno-such-package-xyz-42\n"),
    "vcs_and_include_lines_skipped": check("vcs",
        "-e .\ngit+https://example.invalid/repo.git\n-r other.txt\npytest\n"),
    "comments_only_is_true": check("empty", "# rien ici\n\n"),
    "constraint_loss_marks_false": check("heur", "pytest == =9.9\n"),
}
measured["file_missing_is_false"] = env_checks.check_python_dependencies(str(d / "absent.txt"))
for c in EXAMPLES["deps_cases"]:
    assert measured[c["name"]] is c["verdict"], c["name"]
assert measured["comments_only_is_true"] is True
assert measured["constraint_loss_marks_false"] is False
print("booléens mesurés : vide=True, VCS ignorées, contrainte perdue=False")

booléens mesurés : vide=True, VCS ignorées, contrainte perdue=False


## 5. `map_package_to_module` — exact puis plus long préfixe… **sans frontière de mot**

Le fallback trie les clés par longueur décroissante et teste `startswith` — un test de
*chaîne*, pas de segment pointé : `tests.unit_legacy` commence par `tests.unit`. Et la
clé `""` du mapping attrape **tout** ce qui reste : le fallback documenté `'Autre'` est
**inatteignable**.

In [7]:
cases = EXAMPLES["map_module_cases"]
for c in cases:
    got = project_structure_utils.map_package_to_module(c["package"], c.get("custom"))
    assert got == c["expected"] == c["returns"], c["name"]
trap = [c for c in cases if c["name"] == "prefix_has_no_dotted_boundary"][0]
assert trap["returns"] == "Tests - Unit"
print("mapping mesuré ;", trap["package"], "->", trap["returns"], "(piège startswith)")

mapping mesuré ; tests.unit_legacy -> Tests - Unit (piège startswith)


## 6. Imports — par nom, par chemin, `__init__.py` piégé

`test_module_import_by_path` insère le répertoire parent du fichier dans `sys.path`
**temporairement** et le retire ensuite (effet de bord mesuré). Deux pièges mesurés :
pour un `__init__.py`, le code insère le répertoire du **package** (pas son parent) —
l'import d'un package autonome échoue ; et une erreur de *syntaxe* à l'import n'est pas
une `ImportError` : elle tombe dans le handler générique.

In [8]:
d = Path(tempfile.mkdtemp(prefix="imp_"))
ok, msg = import_testing_utils.test_module_import_by_name("json")
assert ok and msg.startswith("✓ Module 'json' importé avec succès par nom.")
ok, msg = import_testing_utils.test_module_import_by_name("module_inexistant_zz_42")
assert not ok and msg.startswith("✗ Erreur d'importation (ImportError)")

(d / "ok_module.py").write_text("VALUE = 42\n", encoding="utf-8")
ok, msg = import_testing_utils.test_module_import_by_path(d / "ok_module.py")
sys.modules.pop("ok_module", None)
assert ok and msg.startswith("✓ Module 'ok_module' (depuis")
assert str(d.resolve()) not in sys.path, "sys.path must be restored"

pkg = d / "fake_pkg"; pkg.mkdir()
(pkg / "__init__.py").write_text("PKG = 1\n", encoding="utf-8")
ok, msg = import_testing_utils.test_module_import_by_path(pkg / "__init__.py")
sys.modules.pop("fake_pkg", None)
assert not ok  # insère le répertoire du package, pas son parent
assert msg.startswith("✗ Erreur d'importation (ImportError) pour 'fake_pkg'")

(d / "broken_module.py").write_text("def f(:\n    pass\n", encoding="utf-8")
ok, msg = import_testing_utils.test_module_import_by_path(d / "broken_module.py")
sys.modules.pop("broken_module", None)
assert not ok and msg.startswith("✗ Erreur inattendue")

(d / "notes.txt").write_text("no\n", encoding="utf-8")
ok, msg = import_testing_utils.test_module_import_by_path(d / "notes.txt")
assert not ok and msg.startswith("✗ Chemin de module invalide")
assert import_testing_utils.test_import is import_testing_utils.test_module_import_by_name
print("imports mesurés ; sys.path restauré :", str(d.resolve()) not in sys.path)

imports mesurés ; sys.path restauré : True


## 7. Apostrophes de docstrings — alternance triée, idempotence… et pas de `\b`

`fix_docstrings_apostrophes` remplace `d'un` par `"d'un"` via une alternance regex
triée par longueur décroissante. Deux conséquences mesurées : `jusqu'au` (8 car.)
gagne sur `jusqu'à` (7), et l'absence de frontière de mot fait matcher `jusqu'au`
**à l'intérieur** de `jusqu'aubaine`.

In [9]:
d = Path(tempfile.mkdtemp(prefix="apos_"))
p = d / "a.py"; p.write_text("C'est un test d'évaluation simple.\n", encoding="utf-8")
ok1 = format_utils.fix_docstrings_apostrophes(str(p))
after1 = p.read_text(encoding="utf-8")
ok2 = format_utils.fix_docstrings_apostrophes(str(p))
after2 = p.read_text(encoding="utf-8")
assert ok1 and ok2
assert after1 == "\"C'est\" un test \"d'évaluation\" simple.\n"
assert after2 == after1  # idempotent

p = d / "b.py"; p.write_text("jusqu'au bout et jusqu'à la fin\n", encoding="utf-8")
assert format_utils.fix_docstrings_apostrophes(str(p))
assert p.read_text(encoding="utf-8") == "\"jusqu'au\" bout et \"jusqu'à\" la fin\n"

p = d / "c.py"; p.write_text("jusqu'aubaine\n", encoding="utf-8")
assert format_utils.fix_docstrings_apostrophes(str(p))
assert p.read_text(encoding="utf-8") == "\"jusqu'au\"baine\n"

assert format_utils.fix_docstrings_apostrophes(str(d / "nope.py")) is False
cases = {c["name"]: c for c in EXAMPLES["apostrophe_cases"]}
assert cases["basic_quoting"]["content_after_first"] == after1
assert cases["no_word_boundary_overmatch"]["content_after"] == "\"jusqu'au\"baine\n"
print("apostrophes : tri longueur d'abord, idempotent, sur-match sans frontière — mesuré")

apostrophes : tri longueur d'abord, idempotent, sur-match sans frontière — mesuré


## 8. Syntaxe et tokens — deux analyses distinctes

`check_python_syntax` (ast.parse) rend un contexte `>> ligne` autour de l'erreur ;
`check_python_tokens` (tokenize) cherche les `ERRORTOKEN` et attrape la
`TokenError` des chaînes non terminées.

In [10]:
d = Path(tempfile.mkdtemp(prefix="syn_"))
p = d / "ok.py"; p.write_text("def hi():\n    print('hi')\n", encoding="utf-8")
ok, msg, ctx = code_validation.check_python_syntax(str(p))
assert ok and ctx == []

p = d / "broken.py"; p.write_text("def hi():\n    print('oops)\n", encoding="utf-8")
ok, msg, ctx = code_validation.check_python_syntax(str(p))
assert not ok and "Erreur de syntaxe" in msg
marker = [l for l in ctx if l.startswith(">> ")]
assert len(marker) == 1 and "print('oops)" in marker[0]
sc = {c["name"]: c for c in EXAMPLES["syntax_cases"]}["syntax_error_context"]
assert sc["context"] == ctx

ok, msg, ctx = code_validation.check_python_syntax(str(d / "absent.py"))
assert not ok and "Fichier non trouvé" in msg
print("contexte d'erreur :", ctx)

contexte d'erreur : []


In [11]:
p = d / "dollars.py"; p.write_text("x = 1\n$ = 2\n", encoding="utf-8")
ok, msg, errs = code_validation.check_python_tokens(str(p))
assert not ok and len(errs) == 1
assert errs[0]["line"] == 2 and errs[0]["col"] == 0
tc = {c["name"]: c for c in EXAMPLES["syntax_cases"]}
assert tc["error_token_flagged"]["error_tokens"] == errs

p = d / "unterminated.py"; p.write_text("s = '''abc\n", encoding="utf-8")
ok, msg, errs = code_validation.check_python_tokens(str(p))
assert not ok and "Erreur de tokenization" in msg
ut = tc["token_error_unterminated_string"]
assert ut["error_tokens"] == errs
print("ERRORTOKEN et TokenError mesurés :", errs)

ERRORTOKEN et TokenError mesurés : [{'line': 1, 'col': 4, 'message': "('EOF in multi-line string', (1, 4))"}]


## 9. Références de répertoires — exclusions et plafond d'exemples

`analyze_directory_references` marche le projet, **exclut** `docs/`, `venv/`,
`node_modules/`, `target/`… et ne garde que **5 exemples** par motif, tout en comptant tout.

In [12]:
root = Path(tempfile.mkdtemp(prefix="refs_"))
(root / "subdir").mkdir(); (root / "docs").mkdir()
(root / "file1.py").write_text(
    "a = 'config/settings.json'\nb = 'config/other.yaml'\nc = 'data/input.csv'\n",
    encoding="utf-8")
(root / "subdir" / "file2.py").write_text("d = 'data/x'\n", encoding="utf-8")
(root / "file3.txt").write_text("config/ignored\n", encoding="utf-8")
(root / "docs" / "ignored.py").write_text("z = 'config/still-ignored'\n", encoding="utf-8")

res = code_validation.analyze_directory_references(str(root), {
    "config_refs": re.compile(r"config/"), "data_refs": re.compile(r"data/")})
assert {k: v["count"] for k, v in res.items()} == {"config_refs": 2, "data_refs": 2}
assert {Path(f).name for f in res["config_refs"]["files"]} == {"file1.py"}
assert {Path(f).name: n for f, n in res["data_refs"]["files"].items()} == {"file1.py": 1, "file2.py": 1}

root2 = Path(tempfile.mkdtemp(prefix="cap_"))
(root2 / "many.py").write_text(
    "".join(f"line{i} = 'config/f{i}'\n" for i in range(1, 8)), encoding="utf-8")
res2 = code_validation.analyze_directory_references(str(root2), {"c": re.compile(r"config/")})
assert res2["c"]["count"] == 7 and len(res2["c"]["examples"]) == 5
cap = {c["name"]: c for c in EXAMPLES["dir_reference_cases"]}["examples_capped_at_five"]
assert cap["count"] == 7 and cap["n_examples"] == 5
print("7 occurrences comptées,", len(res2['c']['examples']), "exemples conservés")

7 occurrences comptées, 5 exemples conservés


## 10. Couverture — parse XML, historique **fabriqué**

`parse_coverage_xml` multiplie les taux par 100. `create_initial_coverage_history`
écrit deux entrées : la courante, et une **fictive** à J−30 avec line-rate **−5 points**,
lines_covered **×0,9** (tronqué à l'entier). Un historique JSON corrompu est
**silencieusement écrasé**, pas refusé.

In [13]:
d = Path(tempfile.mkdtemp(prefix="cov_"))
xml = d / "coverage.xml"
xml.write_text(
    "<coverage line-rate='0.735' branch-rate='0.5' lines-valid='200' "
    "lines-covered='147' branches-valid='40' branches-covered='20'>"
    "<packages><package name='alpha' line-rate='0.8' branch-rate='0.4'/>"
    "<package name='beta' line-rate='0.612345' branch-rate='0.2'/></packages></coverage>",
    encoding="utf-8")
data = coverage_utils.parse_coverage_xml(xml)
assert data is not None
data.pop("timestamp")
pc = {c["name"]: c for c in EXAMPLES["coverage_cases"]}["parse_ok"]
assert data == pc["data"]
assert data["global_line_rate"] == 73.5
assert coverage_utils.parse_coverage_xml(d / "absent.xml") is None
bad = d / "bad.xml"; bad.write_text("<coverage><packages>", encoding="utf-8")
assert coverage_utils.parse_coverage_xml(bad) is None
print("taux ×100 mesurés :", data["global_line_rate"], data["packages"])

taux ×100 mesurés : 73.5 {'alpha': {'line_rate': 80.0, 'branch_rate': 40.0}, 'beta': {'line_rate': 61.234500000000004, 'branch_rate': 20.0}}


In [14]:
fixed_input = {
    "global_line_rate": 73.5, "global_branch_rate": 50.0,
    "lines_valid": 200, "lines_covered": 57,
    "branches_valid": 10, "branches_covered": 4,
    "packages": {"alpha": {"line_rate": 80.0, "branch_rate": 40.0},
                  "beta": {"line_rate": 61.2, "branch_rate": 20.0}},
    "timestamp": "2026-01-01 12:00:00",
}
hist = d / "hist" / "coverage_history.json"
assert coverage_utils.create_initial_coverage_history(dict(fixed_input), hist)
entries = json.loads(hist.read_text(encoding="utf-8"))
assert len(entries) == 2 and entries[1] == fixed_input
prev = dict(entries[0]); prev_ts = prev.pop("timestamp")
assert isinstance(prev_ts, str)
assert prev["global_line_rate"] == 68.5          # −5 points
assert prev["lines_covered"] == 51               # int(57 × 0.9)
assert prev["packages"]["alpha"]["line_rate"] == 75.0
assert prev["packages"]["beta"]["line_rate"] == 56.2
ih = {c["name"]: c for c in EXAMPLES["coverage_cases"]}["initial_history_fabricates_minus_5pct_30_days_ago"]
assert ih["previous_entry"] == prev and ih["current_entry_equals_input"] is True

h2 = d / "h2.json"
coverage_utils.save_coverage_history({"global_line_rate": 10.0, "timestamp": "2026-01-02 08:00:00"}, h2)
coverage_utils.save_coverage_history({"global_line_rate": 12.0, "timestamp": "2026-01-03 08:00:00"}, h2)
assert [e["global_line_rate"] for e in json.loads(h2.read_text(encoding="utf-8"))] == [10.0, 12.0]

h3 = d / "h3.json"; h3.write_text("definitely-not-json{", encoding="utf-8")
coverage_utils.save_coverage_history({"global_line_rate": 5.0, "timestamp": "2026-01-04 08:00:00"}, h3)
reset = json.loads(h3.read_text(encoding="utf-8"))
assert len(reset) == 1 and reset[0]["global_line_rate"] == 5.0
print("historique : 2 entrées, précédent fabriqué à −5 pts / ×0,9 ; corrompu → reset")

historique : 2 entrées, précédent fabriqué à −5 pts / ×0,9 ; corrompu → reset


## 11. autopep8 — les arguments par défaut remplacés, pas étendus

`format_python_file_with_autopep8` vérifie d'abord le fichier, puis `autopep8 --version`,
puis lance `["autopep8"] + args + [fichier]`. Passer `autopep8_args` **remplace** les
défauts (`--in-place --aggressive --aggressive`) : sans `--in-place`, le fichier n'est
pas modifié. Sous stub pour mesurer la commande exacte.

In [15]:
d = Path(tempfile.mkdtemp(prefix="fmt_"))
def with_stub(file_path, args=None):
    calls = []
    class _Proc: returncode = 0; stdout = ""; stderr = ""
    real_run = code_formatting_utils.subprocess.run
    def fake_run(cmd, **kw):
        calls.append(list(cmd)); return _Proc()
    code_formatting_utils.subprocess.run = fake_run
    try:
        return code_formatting_utils.format_python_file_with_autopep8(file_path, args), calls
    finally:
        code_formatting_utils.subprocess.run = real_run

p = d / "ugly.py"; p.write_text("x=1\n", encoding="utf-8")
ok, calls = with_stub(str(p))
assert ok and calls[0] == ["autopep8", "--version"]
assert calls[1][:4] == ["autopep8", "--in-place", "--aggressive", "--aggressive"]
assert calls[1][-1] == str(p)

ok, calls = with_stub(str(p), ["--diff"])
assert ok and calls[1] == ["autopep8", "--diff", str(p)]

ok, calls = with_stub(str(d / "absent.py"))
assert ok is False and calls == []
ft = {c["name"]: c for c in EXAMPLES["format_tool_cases"]}
assert ft["default_args_are_in_place_double_aggressive"]["format_command_head"] == \
    ["autopep8", "--in-place", "--aggressive", "--aggressive"]
assert ft["missing_file_fails_before_any_subprocess"]["n_subprocess_calls"] == 0
print("commandes mesurées :", calls if calls else "aucun appel si fichier absent")

commandes mesurées : aucun appel si fichier absent


## À retenir

1. **latin-1 n'échoue jamais** — la chaîne d'encodage « réussit » toujours en dernier
   recours ; un vrai diagnostic ne peut pas venir de `fix_file_encoding` seul.
2. **L'environnement Java est un ET** — `java -version` OK sans `JAVA_HOME`/bin/java.exe
   valide rend False.
3. **`check_python_dependencies` rend un booléen** : fichier vide → True, VCS ignorées,
   contrainte perdue → False malgré la récupération du nom.
4. **`map_package_to_module` n'a pas de frontière pointée** — `tests.unit_legacy` →
   `Tests - Unit` (même leçon que la frontière-lettre de #2012).
5. **`sys.path` est un effet de bord** — `test_module_import_by_path` le restaure ;
   une erreur de syntaxe à l'import n'est pas une ImportError.
6. **L'historique de couverture initial est fabriqué** (J−30, −5 pts, ×0,9) et un
   historique corrompu est écrasé en silence — ne pas le lire comme des données réelles.

*Asset mesuré par le builder, rejoué en asserts par le notebook, gardé par
`tests/unit/coursia/dev_tools/test_dev_tools_roundtrip.py`.*